## RAG Pipelines

### Setup Sample Data

This code block initializes sample text documents about **Space Exploration** and **Quantum Computing**. It then writes these texts to corresponding files in the `../data/text_files/` directory, automatically creating the necessary parent directories if they don't already exist.

In [2]:
sample_texts={
    "../data/text_files/space_exploration.txt":"""Space Exploration Basics

Space exploration is the use of astronomy and space technology to explore outer space. Physical exploration of space is conducted both by human spaceflights and by robotic spacecraft. Currently, humanity is focused on returning to the Moon and eventually reaching Mars.

The history of space exploration began with the launch of Sputnik 1 by the Soviet Union in 1957. This triggered the Space Race, culminating in the Apollo 11 mission in 1969, which successfully landed the first humans on the Moon.

In recent years, the commercialization of space has accelerated, with private companies pioneering reusable launch systems, lowering the cost of access to space. Robotic missions like the James Webb Space Telescope continue to push the boundaries of human knowledge by observing distant galaxies, exoplanets, and remnants of the early universe.""",
    
    "../data/text_files/quantum_computing.txt": """Quantum Computing Fundamentals

Quantum computing is a rapidly-emerging technology that harnesses the laws of quantum mechanics to solve problems too complex for classical computers. It utilizes quantum bits, or qubits, allowing for superposition and entanglement, drastically speeding up certain calculations.

Unlike classical bits that must be either a 0 or a 1, a qubit can exist in a superposition of both states simultaneously. Combined with quantum entanglement, this allows quantum computers to process multiple possibilities at an unprecedented scale.

Major technology companies and research institutions are actively developing quantum hardware. These advancements hold the potential to revolutionize fields such as cryptography, drug discovery, materials science, and complex system optimization."""

}

import os
for filepath,content in sample_texts.items():
    # Adding a safety check to automatically make the data directories if missing!
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

#### This code loads all `.txt` files from the specified folder using LangChain’s `TextLoader`, stores them as `Document` objects in a list, and then prints a short preview of each document including its source path and the first part of its text content.

In [3]:
from langchain_community.document_loaders import TextLoader
from pathlib import Path                        

folder = Path("../data/text_files")
all_documents = []

for filename in folder.glob("*.txt"):          
    loader = TextLoader(filename, encoding="utf-8") 
    all_documents.extend(loader.load())  

all_documents

/Users/ahmedarif/Documents/Antigravity/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/space_exploration.txt'}, page_content='Space Exploration Basics\n\nSpace exploration is the use of astronomy and space technology to explore outer space. Physical exploration of space is conducted both by human spaceflights and by robotic spacecraft. Currently, humanity is focused on returning to the Moon and eventually reaching Mars.\n\nThe history of space exploration began with the launch of Sputnik 1 by the Soviet Union in 1957. This triggered the Space Race, culminating in the Apollo 11 mission in 1969, which successfully landed the first humans on the Moon.\n\nIn recent years, the commercialization of space has accelerated, with private companies pioneering reusable launch systems, lowering the cost of access to space. Robotic missions like the James Webb Space Telescope continue to push the boundaries of human knowledge by observing distant galaxies, exoplanets, and remnants of the early universe.'),
 Document(metadata={'source':

### Using DirectoryLoader to Load Multiple Text Files

The following code uses LangChain's `DirectoryLoader` to automatically discover and load all `.txt` files from the `../data/text_files/` directory. It utilizes `TextLoader` under the hood to read the content of each file and creates a single list of `Document` objects.

In [4]:
from langchain_community.document_loaders import DirectoryLoader

# Initialize the DirectoryLoader to load .txt files from the text_files directory
file_path_txt = Path("../data/text_files")
loader_txt = DirectoryLoader(file_path_txt,
                          glob="**/*.txt",
                         loader_cls=TextLoader
                         )

# Load all matching documents
documents_txt = loader_txt.load()

# Print the number of documents loaded and their content preview
print(f"Loaded {len(documents_txt)} documents.")
for doc in documents_txt:
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Content preview: {doc.page_content[:100]}...\n")

Loaded 2 documents.
Source: ../data/text_files/space_exploration.txt
Content preview: Space Exploration Basics

Space exploration is the use of astronomy and space technology to explore ...

Source: ../data/text_files/quantum_computing.txt
Content preview: Quantum Computing Fundamentals

Quantum computing is a rapidly-emerging technology that harnesses th...



In the second block of code, we used `TextLoader` directly. Because `TextLoader` is designed to read only a single file at a time, we had to write a manual Python for loop to search the folder for .txt files, load each file individually, and append them to a list manually.

In the third block of code, we used `DirectoryLoader`. This loader is designed specifically to handle entire folders of files. We simply pointed it to our folder, told it to look for .txt files (using the glob pattern), and instructed it to use TextLoader as the "engine" to read the text.

## loading the `pdf` files

In [5]:
from langchain_community.document_loaders import PyPDFLoader
file_path_pdf = Path("../data/pdf")
loader_pdf = DirectoryLoader(file_path_pdf,
                                glob = "**/*.pdf",
                                loader_cls = PyPDFLoader)

documents_pdf = loader_pdf.load()

for doc in documents_pdf:
    source = Path(doc.metadata['source'])
    doc.metadata['file_name'] = source.name
    doc.metadata['file_type'] = 'pdf'

documents_pdf[0].metadata



{'producer': 'iText® 5.5.13.3 ©2000-2022 iText Group NV (SPRINGER SBM; licensed version)',
 'creator': 'PyPDF',
 'creationdate': '2025-09-18T03:03:54+02:00',
 'moddate': '2025-09-18T03:03:54+02:00',
 'source': '../data/pdf/Retrieval-Augmented_Generation_RAG.pdf',
 'total_pages': 12,
 'page': 0,
 'page_label': '1',
 'file_name': 'Retrieval-Augmented_Generation_RAG.pdf',
 'file_type': 'pdf'}

## Chunking


In [63]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# making a siplitter function

def splitter (documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
                                                   chunk_size = chunk_size,
                                                   chunk_overlap = chunk_overlap,
                                                   length_function = len)
    split_docs = text_splitter.split_documents(documents) 
    print(f"split {len(documents)} documents intto {len(split_docs)} chunks")

    return split_docs


In [64]:
chunks = splitter(documents_pdf)
chunks

split 79 documents intto 325 chunks


[Document(metadata={'producer': 'iText® 5.5.13.3 ©2000-2022 iText Group NV (SPRINGER SBM; licensed version)', 'creator': 'PyPDF', 'creationdate': '2025-09-18T03:03:54+02:00', 'moddate': '2025-09-18T03:03:54+02:00', 'source': '../data/pdf/Retrieval-Augmented_Generation_RAG.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'file_name': 'Retrieval-Augmented_Generation_RAG.pdf', 'file_type': 'pdf'}, page_content='CATCHWORD\nRetrieval-Augmented Generation (RAG)\nMichael Klesel • H. Felix Wittmann\nReceived: 22 July 2024 / Accepted: 7 April 2025 / Published online: 1 June 2025\n/C211The Author(s) 2025\nKeywords Retrieval-augmented generation /C1Artiﬁcial\nintelligence /C1Large language models /C1Information\nretrieval\n1 Introduction\nThe necessity for information is a fundamental aspect of\nhuman nature, and as such, there are ongoing efforts to\nenhance information retrieval with information systems\n(Alavi and Leidner 2001; Alavi et al. 2024). Companies\nare particularly affected by 

## Embedding

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


# Instantiate and test the manager
embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12887.84it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


In [55]:
texts = ["Hello world", "I love programming", "Python is great"]
embeddings = embedding_manager.generate_embeddings(texts)
print(embeddings.shape)  # Should print: (3, 384)

Generating embeddings for 3 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.97it/s]

Generated embeddings with shape: (3, 384)
(3, 384)


## vectore sotre

In [57]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


## Apply the embedding && storing them


In [21]:
# convert the text into embeddings
for doc in chunks:
    if getattr(doc, 'page_content', None) is not None:
        doc.page_content = str(doc.page_content).encode('utf-8', 'ignore').decode('utf-8')

texts = [doc.page_content for doc in chunks if getattr(doc, 'page_content', None) is not None]

embeddings = embedding_manager.generate_embeddings(texts)

# sotore the embedding in the vector database
vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 325 texts...


Batches: 100%|██████████| 11/11 [00:05<00:00,  1.95it/s]


Generated embeddings with shape: (325, 384)
Adding 325 documents to vector store...
Successfully added 325 documents to vector store
Total documents in collection: 325


## Retriever Pipeline From VectorStore

In [22]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []


            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [23]:
rag_retriever

In [25]:
rag_retriever.retrieve("What is rag")

Retrieving documents for query: 'What is rag'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 64.09it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_6dad7874_90',
  'content': 'So what is RAG?\n• RAG systems are specialised tools designed for LLMs to facilitate the \nprocess of gathering and using information.\n• We save our resources in a database (vector DB)\n• We Retrieve what’s relevant \n• We use this information to Augment the Generation of the answer \n• To start, we need to represent the content numerically as embeddings\n12',
  'metadata': {'total_pages': 18,
   'doc_index': 90,
   'source': '../data/pdf/AI-for-Education-RAG.pdf',
   'content_length': 354,
   'page': 11,
   'file_type': 'pdf',
   'page_label': '12',
   'creationdate': '2024-03-04T11:16:13+00:00',
   'file_name': 'AI-for-Education-RAG.pdf',
   'moddate': '2024-03-04T11:16:13+00:00',
   'creator': 'PyPDF',
   'producer': 'PyPDF'},
  'similarity_score': 0.38396209478378296,
  'distance': 0.616037905216217,
  'rank': 1},
 {'id': 'doc_44860670_186',
  'content': 'could clarify its broader trajectory. This survey endeavors to\nfill this gap by mappi